# 02 — Nettoyage et prétraitement des données

## Objectifs
- Charger les fichiers Parquet
- Vérifier les anomalies identifiées lors de l'analyse exploratoire
- Appliquer des règles de nettoyage justifiées
- Contrôler la qualité des données après nettoyage

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
from pathlib import Path

data_dir = Path(r"C:\Users\Msi\Documents\projet_data\data\raw\cicids2017")

parquet_files = sorted(data_dir.glob("*.parquet"))

print("Dossier trouvé :", data_dir.exists())
print("Nombre de fichiers Parquet :", len(parquet_files))

for file in parquet_files:
    print(file.name)

Dossier trouvé : True
Nombre de fichiers Parquet : 8
Benign-Monday-no-metadata.parquet
Botnet-Friday-no-metadata.parquet
Bruteforce-Tuesday-no-metadata.parquet
DDoS-Friday-no-metadata.parquet
DoS-Wednesday-no-metadata.parquet
Infiltration-Thursday-no-metadata.parquet
Portscan-Friday-no-metadata.parquet
WebAttacks-Thursday-no-metadata.parquet


In [3]:
# Vérifier la structure de chaque fichier Parquet

for file in parquet_files:
    df_temp = pd.read_parquet(file)

    print("\n" + "=" * 70)
    print("Fichier :", file.name)
    print("Nombre de lignes :", df_temp.shape[0])
    print("Nombre de colonnes :", df_temp.shape[1])
    print("Colonne Label présente :", "Label" in df_temp.columns)
    print("Nombre de colonnes numériques :",
          df_temp.select_dtypes(include="number").shape[1])

    print("Types de données :")
    print(df_temp.dtypes.value_counts().to_string())

    del df_temp


Fichier : Benign-Monday-no-metadata.parquet
Nombre de lignes : 458831
Nombre de colonnes : 78
Colonne Label présente : True
Nombre de colonnes numériques : 77
Types de données :
int32       27
float32     22
int8        20
int16        6
float64      2
category     1

Fichier : Botnet-Friday-no-metadata.parquet
Nombre de lignes : 176038
Nombre de colonnes : 78
Colonne Label présente : True
Nombre de colonnes numériques : 77
Types de données :
int32       26
float32     22
int8        21
int16        6
float64      2
category     1

Fichier : Bruteforce-Tuesday-no-metadata.parquet
Nombre de lignes : 389714
Nombre de colonnes : 78
Colonne Label présente : True
Nombre de colonnes numériques : 77
Types de données :
int32       25
float32     22
int8        20
int16        7
float64      2
int64        1
category     1

Fichier : DDoS-Friday-no-metadata.parquet
Nombre de lignes : 221264
Nombre de colonnes : 78
Colonne Label présente : True
Nombre de colonnes numériques : 77
Types de donnée

In [4]:
reference_columns = None
schema_errors = []

for file in parquet_files:
    columns = list(pd.read_parquet(file).columns)

    if reference_columns is None:
        reference_columns = columns
    elif set(columns) != set(reference_columns):
        schema_errors.append({
            "Fichier": file.name,
            "Colonnes manquantes": sorted(
                set(reference_columns) - set(columns)
            ),
            "Colonnes supplémentaires": sorted(
                set(columns) - set(reference_columns)
            )
        })

print("Nombre de colonnes dans le premier fichier :",
      len(reference_columns))

if schema_errors:
    print("Des différences de colonnes ont été détectées :")
    display(pd.DataFrame(schema_errors))
else:
    print("Les huit fichiers possèdent les mêmes noms de colonnes.")

print("Colonne Label présente :", "Label" in reference_columns)

Nombre de colonnes dans le premier fichier : 78
Les huit fichiers possèdent les mêmes noms de colonnes.
Colonne Label présente : True


In [5]:
#Contrôler les valeurs manquantes, les infinis et les doublons

quality_results = []

for file in parquet_files:
    df_temp = pd.read_parquet(file)

    numeric_df = df_temp.select_dtypes(include="number")

    infinity_count = int(
        np.isinf(numeric_df.to_numpy()).sum()
    )

    quality_results.append({
        "Fichier": file.name,
        "Lignes": len(df_temp),
        "Colonnes": len(df_temp.columns),
        "Valeurs manquantes": int(df_temp.isna().sum().sum()),
        "Valeurs infinies": infinity_count,
        "Lignes dupliquées": int(df_temp.duplicated().sum()),
        "Labels manquants": int(df_temp["Label"].isna().sum())
    })

    del df_temp

quality_df = pd.DataFrame(quality_results)
display(quality_df)

print("\nTotal des lignes :", quality_df["Lignes"].sum())
print("Total des valeurs manquantes :",
      quality_df["Valeurs manquantes"].sum())
print("Total des valeurs infinies :",
      quality_df["Valeurs infinies"].sum())
print("Total des lignes dupliquées :",
      quality_df["Lignes dupliquées"].sum())

,Fichier,Lignes,Colonnes,Valeurs manquantes,Valeurs infinies,Lignes dupliquées,Labels manquants
0,Benign-Monday-no-metadata.parquet,458831,78,0,0,0,0
1,Botnet-Friday-no-metadata.parquet,176038,78,0,0,0,0
2,Bruteforce-Tuesday-no-metadata.parquet,389714,78,0,0,0,0
3,DDoS-Friday-no-metadata.parquet,221264,78,0,0,0,0
4,DoS-Wednesday-no-metadata.parquet,584991,78,0,0,0,0
5,Infiltration-Thursday-no-metadata.parquet,207630,78,0,0,0,0
6,Portscan-Friday-no-metadata.parquet,119522,78,0,0,0,0
7,WebAttacks-Thursday-no-metadata.parquet,155820,78,0,0,0,0



Total des lignes : 2313810
Total des valeurs manquantes : 0
Total des valeurs infinies : 0
Total des lignes dupliquées : 0


### ***Contrôler les valeurs négatives suspectes***

In [6]:
columns_to_check = [
    "Flow Duration",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd Seg Size Min",
    "Init Fwd Win Bytes",
    "Init Bwd Win Bytes"
]

negative_results = []

for file in parquet_files:
    df_temp = pd.read_parquet(file)

    row = {"Fichier": file.name}

    for col in columns_to_check:
        if col in df_temp.columns:
            row[col] = int((df_temp[col] < 0).sum())
        else:
            row[col] = "Colonne absente"

    negative_results.append(row)
    del df_temp

negative_df = pd.DataFrame(negative_results)
display(negative_df)

,Fichier,Flow Duration,Flow IAT Mean,Flow IAT Max,Flow Bytes/s,Flow Packets/s,Fwd Header Length,Bwd Header Length,Fwd Seg Size Min,Init Fwd Win Bytes,Init Bwd Win Bytes
0,Benign-Monday-no-metadata.parquet,15,15,15,9,15,23,17,23,210416,249419
1,Botnet-Friday-no-metadata.parquet,9,9,9,8,9,0,0,0,92084,106692
2,Bruteforce-Tuesday-no-metadata.parquet,16,16,16,0,16,11,5,11,190187,221174
3,DDoS-Friday-no-metadata.parquet,2,2,2,2,2,0,0,0,31099,84321
4,DoS-Wednesday-no-metadata.parquet,21,21,21,16,21,1,0,1,190622,240012
5,Infiltration-Thursday-no-metadata.parquet,4,4,4,4,4,0,0,0,98480,115921
6,Portscan-Friday-no-metadata.parquet,10,10,10,10,10,0,0,0,58510,69551
7,WebAttacks-Thursday-no-metadata.parquet,11,11,11,10,11,0,0,0,79033,92322


In [7]:
from pathlib import Path

# Dossier de destination des données nettoyées
processed_dir = Path(
    r"C:\Users\Msi\Documents\projet_data\data\processed"
)

# Vérifier que le dossier existe
print("Chemin des données nettoyées :", processed_dir)
print("Dossier existant :", processed_dir.exists())


Chemin des données nettoyées : C:\Users\Msi\Documents\projet_data\data\processed
Dossier existant : True


### ***Nettoyage de chaque fichier et enregistrer une copie***

In [8]:
length_columns = [
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd Seg Size Min"
]

cleaning_results = []

for file in parquet_files:
    print(f"\nTraitement de : {file.name}")

    df_temp = pd.read_parquet(file)

    # 1. Remplacer les valeurs infinies dans les colonnes numériques
    numeric_cols = df_temp.select_dtypes(include="number").columns

    infinity_before = int(
        np.isinf(df_temp[numeric_cols].to_numpy()).sum()
    )

    df_temp[numeric_cols] = df_temp[numeric_cols].replace(
        [np.inf, -np.inf], np.nan
    )

    # 2. Remplacer les valeurs négatives des colonnes ciblées
    #    par NaN, sans supprimer les lignes
    negative_before = {}

    for col in length_columns:
        if col in df_temp.columns:
            negative_before[col] = int((df_temp[col] < 0).sum())
            df_temp[col] = df_temp[col].mask(
                df_temp[col] < 0, np.nan
            )

    # 3. Enregistrer dans le dossier de sortie
    output_file = processed_dir / file.name
    df_temp.to_parquet(output_file, index=False)

    cleaning_results.append({
        "Fichier": file.name,
        "Lignes conservées": len(df_temp),
        "Infinis remplacés": infinity_before,
        "Fwd Header Length négatifs": negative_before.get(
            "Fwd Header Length", 0
        ),
        "Bwd Header Length négatifs": negative_before.get(
            "Bwd Header Length", 0
        ),
        "Fwd Seg Size Min négatifs": negative_before.get(
            "Fwd Seg Size Min", 0
        ),
        "Valeurs manquantes après nettoyage": int(
            df_temp.isna().sum().sum()
        )
    })

    del df_temp

cleaning_df = pd.DataFrame(cleaning_results)
display(cleaning_df)

print("\nCopies nettoyées enregistrées dans :", processed_dir)


Traitement de : Benign-Monday-no-metadata.parquet

Traitement de : Botnet-Friday-no-metadata.parquet

Traitement de : Bruteforce-Tuesday-no-metadata.parquet

Traitement de : DDoS-Friday-no-metadata.parquet

Traitement de : DoS-Wednesday-no-metadata.parquet

Traitement de : Infiltration-Thursday-no-metadata.parquet

Traitement de : Portscan-Friday-no-metadata.parquet

Traitement de : WebAttacks-Thursday-no-metadata.parquet


,Fichier,Lignes conservées,Infinis remplacés,Fwd Header Length négatifs,Bwd Header Length négatifs,Fwd Seg Size Min négatifs,Valeurs manquantes après nettoyage
0,Benign-Monday-no-metadata.parquet,458831,0,23,17,23,63
1,Botnet-Friday-no-metadata.parquet,176038,0,0,0,0,0
2,Bruteforce-Tuesday-no-metadata.parquet,389714,0,11,5,11,27
3,DDoS-Friday-no-metadata.parquet,221264,0,0,0,0,0
4,DoS-Wednesday-no-metadata.parquet,584991,0,1,0,1,2
5,Infiltration-Thursday-no-metadata.parquet,207630,0,0,0,0,0
6,Portscan-Friday-no-metadata.parquet,119522,0,0,0,0,0
7,WebAttacks-Thursday-no-metadata.parquet,155820,0,0,0,0,0



Copies nettoyées enregistrées dans : C:\Users\Msi\Documents\projet_data\data\processed


### ***Vérifier les fichiers nettoyés : vérifier qu'il ne reste plus de valeurs négatives dans les trois colonnes ciblées.***

In [9]:
verification_results = []

for original_file in parquet_files:
    cleaned_file = processed_dir / original_file.name

    if not cleaned_file.exists():
        print("Fichier nettoyé manquant :", cleaned_file.name)
        continue

    df_clean = pd.read_parquet(cleaned_file)

    verification_results.append({
        "Fichier": cleaned_file.name,
        "Lignes": len(df_clean),
        "Colonnes": len(df_clean.columns),
        "Valeurs manquantes": int(df_clean.isna().sum().sum()),
        "Label présent": "Label" in df_clean.columns,
        "Fwd Header Length négatifs": int(
            (df_clean["Fwd Header Length"] < 0).sum()
        ),
        "Bwd Header Length négatifs": int(
            (df_clean["Bwd Header Length"] < 0).sum()
        ),
        "Fwd Seg Size Min négatifs": int(
            (df_clean["Fwd Seg Size Min"] < 0).sum()
        )
    })

    del df_clean

verification_df = pd.DataFrame(verification_results)
display(verification_df)

,Fichier,Lignes,Colonnes,Valeurs manquantes,Label présent,Fwd Header Length négatifs,Bwd Header Length négatifs,Fwd Seg Size Min négatifs
0,Benign-Monday-no-metadata.parquet,458831,78,63,True,0,0,0
1,Botnet-Friday-no-metadata.parquet,176038,78,0,True,0,0,0
2,Bruteforce-Tuesday-no-metadata.parquet,389714,78,27,True,0,0,0
3,DDoS-Friday-no-metadata.parquet,221264,78,0,True,0,0,0
4,DoS-Wednesday-no-metadata.parquet,584991,78,2,True,0,0,0
5,Infiltration-Thursday-no-metadata.parquet,207630,78,0,True,0,0,0
6,Portscan-Friday-no-metadata.parquet,119522,78,0,True,0,0,0
7,WebAttacks-Thursday-no-metadata.parquet,155820,78,0,True,0,0,0


In [10]:
#vérification 
for file in sorted(processed_dir.glob("*.parquet")):
    df_check = pd.read_parquet(file)

    print("\nFichier :", file.name)
    print("Dimensions :", df_check.shape)
    print("Label présent :", "Label" in df_check.columns)
    print("Nombre de valeurs manquantes :", df_check.isna().sum().sum())

    cols = [
        "Fwd Header Length",
        "Bwd Header Length",
        "Fwd Seg Size Min"
    ]

    print("Valeurs négatives restantes :")
    print((df_check[cols] < 0).sum().to_dict())


Fichier : Benign-Monday-no-metadata.parquet
Dimensions : (458831, 78)
Label présent : True
Nombre de valeurs manquantes : 63
Valeurs négatives restantes :
{'Fwd Header Length': 0, 'Bwd Header Length': 0, 'Fwd Seg Size Min': 0}

Fichier : Botnet-Friday-no-metadata.parquet
Dimensions : (176038, 78)
Label présent : True
Nombre de valeurs manquantes : 0
Valeurs négatives restantes :
{'Fwd Header Length': 0, 'Bwd Header Length': 0, 'Fwd Seg Size Min': 0}

Fichier : Bruteforce-Tuesday-no-metadata.parquet
Dimensions : (389714, 78)
Label présent : True
Nombre de valeurs manquantes : 27
Valeurs négatives restantes :
{'Fwd Header Length': 0, 'Bwd Header Length': 0, 'Fwd Seg Size Min': 0}

Fichier : DDoS-Friday-no-metadata.parquet
Dimensions : (221264, 78)
Label présent : True
Nombre de valeurs manquantes : 0
Valeurs négatives restantes :
{'Fwd Header Length': 0, 'Bwd Header Length': 0, 'Fwd Seg Size Min': 0}

Fichier : DoS-Wednesday-no-metadata.parquet
Dimensions : (584991, 78)
Label présent : 

### ***vérifions les valeurs manquants pour la meilleur choix du modéle***

In [11]:

missing_results = []

for file in sorted(processed_dir.glob("*.parquet")):
    df = pd.read_parquet(file)

    missing_by_column = df.isna().sum()
    missing_columns = missing_by_column[missing_by_column > 0]

    for column, count in missing_columns.items():
        missing_results.append({
            "Fichier": file.name,
            "Colonne": column,
            "Valeurs manquantes": int(count),
            "Label concerné": column == "Label"
        })

missing_df = pd.DataFrame(missing_results)

if missing_df.empty:
    print("Aucune valeur manquante détectée.")
else:
    display(missing_df)


,Fichier,Colonne,Valeurs manquantes,Label concerné
0,Benign-Monday-no-metadata.parquet,Fwd Header Length,23,False
1,Benign-Monday-no-metadata.parquet,Bwd Header Length,17,False
2,Benign-Monday-no-metadata.parquet,Fwd Seg Size Min,23,False
3,Bruteforce-Tuesday-no-metadata.parquet,Fwd Header Length,11,False
4,Bruteforce-Tuesday-no-metadata.parquet,Bwd Header Length,5,False
5,Bruteforce-Tuesday-no-metadata.parquet,Fwd Seg Size Min,11,False
6,DoS-Wednesday-no-metadata.parquet,Fwd Header Length,1,False
7,DoS-Wednesday-no-metadata.parquet,Fwd Seg Size Min,1,False


In [12]:
#nous allons remplacer ces NaN par 0


length_columns = [
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd Seg Size Min"
]

imputation_results = []

for file in sorted(processed_dir.glob("*.parquet")):
    df = pd.read_parquet(file)

    missing_before = int(df[length_columns].isna().sum().sum())

    df[length_columns] = df[length_columns].fillna(0)

    missing_after = int(df[length_columns].isna().sum().sum())

    # Réenregistrer le fichier nettoyé
    df.to_parquet(file, index=False)

    imputation_results.append({
        "Fichier": file.name,
        "Valeurs manquantes avant": missing_before,
        "Valeurs manquantes après": missing_after,
        "Nombre de lignes": len(df),
        "Label présent": "Label" in df.columns
    })

    del df

imputation_df = pd.DataFrame(imputation_results)
display(imputation_df)


,Fichier,Valeurs manquantes avant,Valeurs manquantes après,Nombre de lignes,Label présent
0,Benign-Monday-no-metadata.parquet,63,0,458831,True
1,Botnet-Friday-no-metadata.parquet,0,0,176038,True
2,Bruteforce-Tuesday-no-metadata.parquet,27,0,389714,True
3,DDoS-Friday-no-metadata.parquet,0,0,221264,True
4,DoS-Wednesday-no-metadata.parquet,2,0,584991,True
5,Infiltration-Thursday-no-metadata.parquet,0,0,207630,True
6,Portscan-Friday-no-metadata.parquet,0,0,119522,True
7,WebAttacks-Thursday-no-metadata.parquet,0,0,155820,True


In [13]:

verification_finale = []

for file in sorted(processed_dir.glob("*.parquet")):
    df = pd.read_parquet(file)

    verification_finale.append({
        "Fichier": file.name,
        "Lignes": len(df),
        "Colonnes": len(df.columns),
        "Valeurs manquantes totales": int(df.isna().sum().sum()),
        "Label présent": "Label" in df.columns
    })

    del df

verification_df = pd.DataFrame(verification_finale)
display(verification_df)


,Fichier,Lignes,Colonnes,Valeurs manquantes totales,Label présent
0,Benign-Monday-no-metadata.parquet,458831,78,0,True
1,Botnet-Friday-no-metadata.parquet,176038,78,0,True
2,Bruteforce-Tuesday-no-metadata.parquet,389714,78,0,True
3,DDoS-Friday-no-metadata.parquet,221264,78,0,True
4,DoS-Wednesday-no-metadata.parquet,584991,78,0,True
5,Infiltration-Thursday-no-metadata.parquet,207630,78,0,True
6,Portscan-Friday-no-metadata.parquet,119522,78,0,True
7,WebAttacks-Thursday-no-metadata.parquet,155820,78,0,True


In [14]:
#Vérifier les doublons


duplicate_results = []

for file in sorted(processed_dir.glob("*.parquet")):
    df = pd.read_parquet(file)

    duplicate_results.append({
        "Fichier": file.name,
        "Nombre de lignes": len(df),
        "Lignes dupliquées": int(df.duplicated().sum()),
        "Pourcentage dupliqué": round(
            df.duplicated().mean() * 100, 4
        )
    })

    del df

duplicates_df = pd.DataFrame(duplicate_results)
display(duplicates_df)


,Fichier,Nombre de lignes,Lignes dupliquées,Pourcentage dupliqué
0,Benign-Monday-no-metadata.parquet,458831,0,0.0
1,Botnet-Friday-no-metadata.parquet,176038,0,0.0
2,Bruteforce-Tuesday-no-metadata.parquet,389714,0,0.0
3,DDoS-Friday-no-metadata.parquet,221264,0,0.0
4,DoS-Wednesday-no-metadata.parquet,584991,0,0.0
5,Infiltration-Thursday-no-metadata.parquet,207630,0,0.0
6,Portscan-Friday-no-metadata.parquet,119522,0,0.0
7,WebAttacks-Thursday-no-metadata.parquet,155820,0,0.0


In [15]:
#Vérifier et uniformiser les étiquettes



label_results = []

for file in sorted(processed_dir.glob("*.parquet")):
    df = pd.read_parquet(file, columns=["Label"])

    counts = df["Label"].value_counts(dropna=False)

    for label, count in counts.items():
        label_results.append({
            "Fichier": file.name,
            "Label": str(label),
            "Nombre de lignes": int(count)
        })

    del df

labels_df = pd.DataFrame(label_results)
display(labels_df.sort_values(["Fichier", "Label"]))


,Fichier,Label,Nombre de lignes
0,Benign-Monday-no-metadata.parquet,Benign,458831
1,Botnet-Friday-no-metadata.parquet,Benign,174601
2,Botnet-Friday-no-metadata.parquet,Bot,1437
3,Bruteforce-Tuesday-no-metadata.parquet,Benign,380564
4,Bruteforce-Tuesday-no-metadata.parquet,FTP-Patator,5931
5,Bruteforce-Tuesday-no-metadata.parquet,SSH-Patator,3219
7,DDoS-Friday-no-metadata.parquet,Benign,93250
6,DDoS-Friday-no-metadata.parquet,DDoS,128014
8,DoS-Wednesday-no-metadata.parquet,Benign,391235
10,DoS-Wednesday-no-metadata.parquet,DoS GoldenEye,10286


In [17]:
#Corriger les étiquettes mal encodées


label_corrections = {
    "Web Attack � Brute Force": "Web Attack Brute Force",
    "Web Attack � XSS": "Web Attack XSS",
    "Web Attack � Sql Injection": "Web Attack Sql Injection"
}

web_file = processed_dir / "WebAttacks-Thursday-no-metadata.parquet"

# Recharger le fichier nettoyé
df_web = pd.read_parquet(web_file)

# Convertir Label en texte pour autoriser les remplacements
df_web["Label"] = df_web["Label"].astype(str)

print("Labels avant correction :")
print(df_web["Label"].value_counts())

# Remplacer uniquement les trois labels concernés
df_web["Label"] = df_web["Label"].replace(label_corrections)

print("\nLabels après correction :")
print(df_web["Label"].value_counts())

# Enregistrer le fichier corrigé
df_web.to_parquet(web_file, index=False)

print("\nFichier WebAttacks corrigé et enregistré.")


Labels avant correction :
Label
Benign                        153677
Web Attack � Brute Force        1470
Web Attack � XSS                 652
Web Attack � Sql Injection        21
Name: count, dtype: int64

Labels après correction :
Label
Benign                      153677
Web Attack Brute Force        1470
Web Attack XSS                 652
Web Attack Sql Injection        21
Name: count, dtype: int64

Fichier WebAttacks corrigé et enregistré.


In [19]:
#Vérifier le résultat

df_web = pd.read_parquet(web_file)

print("Labels contenant encore le caractère de remplacement :")
print(
    df_web.loc[
        df_web["Label"].astype(str).str.contains("�", regex=False),
        "Label"
    ].unique()
)

print("\nRépartition finale des labels :")
display(df_web["Label"].value_counts())


Labels contenant encore le caractère de remplacement :
<ArrowStringArray>
[]
Length: 0, dtype: str

Répartition finale des labels :


Label
Benign                      153677
Web Attack Brute Force        1470
Web Attack XSS                 652
Web Attack Sql Injection        21
Name: count, dtype: int64

### ***Fusionner les fichiers nettoyés***

In [20]:

# Récupérer les 8 fichiers nettoyés
clean_files = sorted(processed_dir.glob("*.parquet"))

print("Nombre de fichiers trouvés :", len(clean_files))

# Charger et fusionner les fichiers
dfs = [pd.read_parquet(file) for file in clean_files]

df_all = pd.concat(dfs, ignore_index=True)

del dfs

print("\nDimensions du dataset fusionné :", df_all.shape)
print("Nombre total de lignes :", len(df_all))
print("Nombre de colonnes :", len(df_all.columns))

print("\nVérification de la colonne Label :")
print("Label présent :", "Label" in df_all.columns)
print("Nombre de classes :", df_all["Label"].nunique())


Nombre de fichiers trouvés : 8

Dimensions du dataset fusionné : (2313810, 78)
Nombre total de lignes : 2313810
Nombre de colonnes : 78

Vérification de la colonne Label :
Label présent : True
Nombre de classes : 15


#### Examiner la répartition globale des classes

In [23]:
label_counts = df_all["Label"].value_counts()

label_summary = pd.DataFrame({
    "Nombre de flux": label_counts,
    "Pourcentage (%)": (
        label_counts / len(df_all) * 100
    ).round(4)
})

display(label_summary)


# Vérifier les doublons dans le dataset fusionné
nombre_doublons = int(df_all.duplicated().sum())

print("Nombre de lignes dupliquées :", nombre_doublons)
print(
    "Pourcentage de doublons :",
    round(nombre_doublons / len(df_all) * 100, 4),
    "%"
)


,Nombre de flux,Pourcentage (%)
Label,,
Benign,1977318,85.4572
DoS Hulk,172846,7.4702
DDoS,128014,5.5326
DoS GoldenEye,10286,0.4445
FTP-Patator,5931,0.2563
DoS slowloris,5385,0.2327
DoS Slowhttptest,5228,0.2259
SSH-Patator,3219,0.1391
PortScan,1956,0.0845


Nombre de lignes dupliquées : 82004
Pourcentage de doublons : 3.5441 %


### Vérifier les doublons et les étiquettes

In [25]:

# Colonnes utilisées comme caractéristiques (sans Label)
feature_cols = [col for col in df_all.columns if col != "Label"]

# Repérer les lignes qui ont les mêmes caractéristiques
# même si leur Label est éventuellement différent
duplicate_features = df_all.duplicated(
    subset=feature_cols,
    keep=False
)

df_duplicate_features = df_all.loc[duplicate_features]

print(
    "Nombre de lignes appartenant à un groupe de caractéristiques dupliquées :",
    len(df_duplicate_features)
)

# Vérifier si des caractéristiques identiques ont plusieurs labels
if not df_duplicate_features.empty:
    labels_per_features = (
        df_duplicate_features
        .groupby(feature_cols, dropna=False)["Label"]
        .nunique()
    )

    conflicting_groups = labels_per_features[labels_per_features > 1]

    print(
        "Groupes de caractéristiques associés à plusieurs labels :",
        len(conflicting_groups)
    )
else:
    print("Aucun groupe de caractéristiques dupliquées.")


Nombre de lignes appartenant à un groupe de caractéristiques dupliquées : 131890
Groupes de caractéristiques associés à plusieurs labels : 270


### examiner les conflits de labels

In [27]:

# Afficher les groupes de caractéristiques identiques
# associés à plusieurs labels

if not conflicting_groups.empty:
    print("Nombre de groupes conflictuels :", len(conflicting_groups))

    # Afficher les 10 premiers groupes conflictuels
    exemples = conflicting_groups.head(10).reset_index()
    print("\nExemples de caractéristiques conflictuelles :")
    display(exemples)

    # Récupérer les lignes correspondant à ces groupes
    cles = conflicting_groups.head(10).index

    for numero, cle in enumerate(cles, start=1):
        masque = pd.Series(True, index=df_all.index)

        for colonne, valeur in zip(feature_cols, cle if isinstance(cle, tuple) else (cle,)):
            masque &= df_all[colonne].eq(valeur)

        lignes = df_all.loc[masque, ["Label"]].value_counts().reset_index(name="Nombre")
        print(f"\nGroupe {numero} — labels associés :")
        display(lignes)
else:
    print("Aucun conflit de labels à examiner.")


Nombre de groupes conflictuels : 270

Exemples de caractéristiques conflictuelles :


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,1,1,1,2,6,2,2,2.0,0.0,...,24.0,0.0,0.0,0,0,0.0,0.0,0,0,2
1,6,1,1,1,6,6,6,6,6.0,0.0,...,20.0,0.0,0.0,0,0,0.0,0.0,0,0,2
2,6,1,2,0,0,0,0,0,0.0,0.0,...,32.0,0.0,0.0,0,0,0.0,0.0,0,0,2
3,6,1,2,0,0,0,0,0,0.0,0.0,...,32.0,0.0,0.0,0,0,0.0,0.0,0,0,2
4,6,2,1,1,2,6,2,2,2.0,0.0,...,24.0,0.0,0.0,0,0,0.0,0.0,0,0,2
5,6,2,1,1,6,6,6,6,6.0,0.0,...,20.0,0.0,0.0,0,0,0.0,0.0,0,0,2
6,6,3,1,1,2,6,2,2,2.0,0.0,...,24.0,0.0,0.0,0,0,0.0,0.0,0,0,2
7,6,3,1,1,6,6,6,6,6.0,0.0,...,20.0,0.0,0.0,0,0,0.0,0.0,0,0,2
8,6,4,1,1,2,6,2,2,2.0,0.0,...,24.0,0.0,0.0,0,0,0.0,0.0,0,0,2
9,6,4,1,1,6,6,6,6,6.0,0.0,...,20.0,0.0,0.0,0,0,0.0,0.0,0,0,2



Groupe 1 — labels associés :


,Label,Nombre
0,Benign,1
1,PortScan,1



Groupe 2 — labels associés :


,Label,Nombre
0,Bot,1
1,Benign,1


... (12 outputs left out)


Groupe 9 — labels associés :


,Label,Nombre
0,Benign,1
1,PortScan,1



Groupe 10 — labels associés :


,Label,Nombre
0,DDoS,1
1,Benign,1


### mesurer précisément les conflits

In [28]:

# 1. Compter les groupes conflictuels selon leurs combinaisons de labels

resume_conflits = (
    df_duplicate_features
    .groupby(feature_cols, dropna=False)["Label"]
    .agg(
        nombre_lignes="size",
        nombre_labels_differents="nunique",
        labels=lambda s: " | ".join(sorted(s.astype(str).unique()))
    )
    .reset_index()
)

# Conserver uniquement les groupes ayant plusieurs labels
resume_conflits = resume_conflits[
    resume_conflits["nombre_labels_differents"] > 1
].copy()

print("Nombre de groupes conflictuels :", len(resume_conflits))
print(
    "Nombre total de lignes dans ces groupes :",
    int(resume_conflits["nombre_lignes"].sum())
)

# 2. Afficher les combinaisons de labels les plus fréquentes
display(
    resume_conflits["labels"]
    .value_counts()
    .head(20)
    .rename_axis("Combinaison de labels")
    .reset_index(name="Nombre de groupes")
)

# 3. Afficher quelques groupes avec leur nombre de lignes
display(
    resume_conflits[
        ["nombre_lignes", "nombre_labels_differents", "labels"]
    ]
    .sort_values("nombre_lignes", ascending=False)
    .head(20)
)


Nombre de groupes conflictuels : 270
Nombre total de lignes dans ces groupes : 566


,Combinaison de labels,Nombre de groupes
0,Benign | DoS Hulk,134
1,Benign | PortScan,106
2,Benign | Bot,25
3,Benign | DDoS,3
4,Benign | DoS slowloris,2


,nombre_lignes,nombre_labels_differents,labels
19781,4,2,Benign | DoS Hulk
11036,3,2,Benign | Bot
648,3,2,Benign | Bot
112,3,2,Benign | DoS Hulk
12960,3,2,Benign | Bot
9258,3,2,Benign | DoS Hulk
11835,3,2,Benign | Bot
139,3,2,Benign | DoS Hulk
16648,3,2,Benign | DoS Hulk
16871,3,2,Benign | DoS Hulk


### mesurer les doublons exacts

In [29]:

# Compter les doublons strictement identiques :
# les 77 caractéristiques ET le Label doivent être identiques.

nombre_doublons_exacts = int(df_all.duplicated(keep="first").sum())

print("Nombre de lignes en double exact :", nombre_doublons_exacts)

print(
    "Pourcentage du jeu de données :",
    round(nombre_doublons_exacts / len(df_all) * 100, 4),
    "%"
)

# Vérifier combien de lignes resteraient après suppression
# d'une occurrence de chaque ligne strictement identique.
nombre_lignes_uniques = len(df_all.drop_duplicates())

print("Nombre de lignes avant :", len(df_all))
print("Nombre de lignes après :", nombre_lignes_uniques)


Nombre de lignes en double exact : 82004
Pourcentage du jeu de données : 3.5441 %
Nombre de lignes avant : 2313810
Nombre de lignes après : 2231806


### supprimer les doubllons exacte 



In [30]:

# Conserver une seule occurrence de chaque ligne entièrement identique
df_all_clean = df_all.drop_duplicates(keep="first").copy()

# Vérifications
print("Dimensions avant :", df_all.shape)
print("Dimensions après :", df_all_clean.shape)
print("Doublons restants :", df_all_clean.duplicated().sum())
print("Valeurs manquantes :", df_all_clean.isna().sum().sum())
print("Nombre de classes :", df_all_clean["Label"].nunique())

# Vérifier la distribution des classes
display(
    df_all_clean["Label"]
    .value_counts()
    .rename_axis("Label")
    .reset_index(name="Nombre")
)


Dimensions avant : (2313810, 78)
Dimensions après : (2231806, 78)
Doublons restants : 0
Valeurs manquantes : 0
Nombre de classes : 15


,Label,Nombre
0,Benign,1895314
1,DoS Hulk,172846
2,DDoS,128014
3,DoS GoldenEye,10286
4,FTP-Patator,5931
5,DoS slowloris,5385
6,DoS Slowhttptest,5228
7,SSH-Patator,3219
8,PortScan,1956
9,Web Attack Brute Force,1470


### Séparer les caractéristiques et la cible

In [31]:

# Séparer les caractéristiques (X) et la cible (y)

X = df_all_clean.drop(columns=["Label"])
y = df_all_clean["Label"].astype(str)

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
print("\nNombre de caractéristiques :", X.shape[1])
print("Nombre de classes :", y.nunique())

# Vérifier les types de données
print("\nTypes de données :")
display(X.dtypes.value_counts())

# Vérifier les valeurs infinies
print(
    "\nNombre de valeurs infinies :",
    int(np.isinf(X.select_dtypes(include=np.number)).sum().sum())
)

# Vérifier les valeurs manquantes
print("Valeurs manquantes dans X :", int(X.isna().sum().sum()))


Dimensions de X : (2231806, 77)
Dimensions de y : (2231806,)

Nombre de caractéristiques : 77
Nombre de classes : 15

Types de données :


int32      24
float32    22
int8       19
int16       7
float64     5
Name: count, dtype: int64


Nombre de valeurs infinies : 0
Valeurs manquantes dans X : 0


### Préparer les ensembles d'entraînement et de test

In [32]:

from sklearn.model_selection import GroupShuffleSplit

# Construire un identifiant de groupe pour chaque ligne :
# les lignes ayant les mêmes caractéristiques appartiennent
# au même groupe, même si leurs labels sont différents.
groupes = pd.util.hash_pandas_object(X, index=False).to_numpy()

# Première séparation : 80 % entraînement, 20 % test
separateur = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

indices_train, indices_test = next(
    separateur.split(X, y, groups=groupes)
)

X_train = X.iloc[indices_train].copy()
X_test = X.iloc[indices_test].copy()
y_train = y.iloc[indices_train].copy()
y_test = y.iloc[indices_test].copy()

print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_test  :", X_test.shape)
print("Dimensions de y_train :", y_train.shape)
print("Dimensions de y_test  :", y_test.shape)

print("\nRépartition des classes dans le test :")
display(y_test.value_counts().to_frame("Nombre"))


Dimensions de X_train : (1785450, 77)
Dimensions de X_test  : (446356, 77)
Dimensions de y_train : (1785450,)
Dimensions de y_test  : (446356,)

Répartition des classes dans le test :


,Nombre
Label,
Benign,378902
DoS Hulk,34735
DDoS,25561
DoS GoldenEye,2033
FTP-Patator,1185
DoS slowloris,1122
DoS Slowhttptest,1066
SSH-Patator,615
PortScan,409
